# Consume a Registered Model on Serverless Spark

Resolve the registered Titanic Spark ML model, reuse its checksum-verified ADLS copy, and run [titanic-serverless-model-consumption.yaml](./titanic-serverless-model-consumption.yaml) as a **standalone managed Spark 3.5 job**.

This notebook is a Python SDK client: model loading and `PipelineModel.transform` execute remotely on serverless Spark, not in the local notebook kernel. Unlike notebook 03, it has no pipeline wrapper. It does not deploy or invoke an endpoint, start AML compute, or retrain/register a model.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import sys
import uuid

import pandas as pd
from azure.ai.ml import MLClient, load_job
from azure.ai.ml.constants import AssetTypes
from azure.identity import AzureCliCredential
from azure.storage.filedatalake import DataLakeServiceClient
from dotenv import load_dotenv
from IPython.display import display


def find_workshop_root() -> Path:
    notebook_file = globals().get("__vsc_ipynb_file__")
    start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AGENDA.md").is_file() and (candidate / ".env.example").is_file():
            return candidate
        nested = candidate / "workshop"
        if (nested / "AGENDA.md").is_file() and (nested / ".env.example").is_file():
            return nested
    raise FileNotFoundError("Run this notebook from inside the workshop folder")


def require_env(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set in workshop/.env")
    return value


def env_flag(name: str) -> bool:
    value = os.getenv(name, "false").strip().lower()
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be exactly true or false")
    return value == "true"


WORKSHOP_ROOT = find_workshop_root()
MODULE_ROOT = WORKSHOP_ROOT / "notebooks" / "06_spark_jobs_pipelines"
SOURCE_ROOT = MODULE_ROOT / "src"
JOB_PATH = MODULE_ROOT / "titanic-serverless-model-consumption.yaml"
OUTPUT_ROOT = WORKSHOP_ROOT / "outputs" / "titanic_model_consumption"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from inference_verification import (
    assert_native_spark_job, assert_prediction_parity,
    download_predictions, load_verified_predictions,
)
from registered_model_artifacts import stage_registered_model
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

SUBSCRIPTION_ID = require_env("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = require_env("AZURE_RESOURCE_GROUP")
WORKSPACE_NAME = require_env("AZUREML_WORKSPACE_NAME")
MODEL_NAME = require_env("TITANIC_SPARK_MODEL_NAME")
MODEL_VERSION = require_env("TITANIC_SPARK_MODEL_VERSION")
STORAGE_ACCOUNT = require_env("TITANIC_SPARK_STORAGE_ACCOUNT")
FILESYSTEM = require_env("TITANIC_SPARK_FILESYSTEM")
OUTPUT_PREFIX = require_env("TITANIC_SPARK_MODEL_CONSUMPTION_OUTPUT_PATH").strip("/")
EXISTING_JOB_NAME = os.getenv("TITANIC_SPARK_MODEL_CONSUMPTION_JOB_NAME", "").strip()
REFERENCE_PATH = os.getenv("TITANIC_SPARK_MODEL_CONSUMPTION_REFERENCE_PREDICTIONS_PATH", "").strip()
RUN_CONSUMPTION = env_flag("RUN_TITANIC_SPARK_MODEL_CONSUMPTION")

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None, process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
registered_model = ml_client.models.get(MODEL_NAME, version=MODEL_VERSION)
if registered_model.type != AssetTypes.CUSTOM_MODEL or not registered_model.path:
    raise TypeError("Expected a registered native Spark custom model artifact")
input_uri = f"abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/data/titanic.csv"

print(f"Workspace: {WORKSPACE_NAME}")
print(f"Registered model: {registered_model.name}:{registered_model.version}")
print(f"Registered artifact: {registered_model.path}")
print(f"Run enabled: {RUN_CONSUMPTION}; existing job: {EXISTING_JOB_NAME or 'none'}")

In [ ]:
spark_job = load_job(JOB_PATH)
if spark_job.type != "spark" or spark_job.compute is not None:
    raise ValueError("Consumption must be a standalone serverless Spark job")
if str(spark_job.resources.runtime_version) != "3.5":
    raise ValueError("Consumption requires managed Spark runtime 3.5")
if spark_job.args.count("__REGISTERED_MODEL_ID__") != 1:
    raise ValueError("The Spark definition must contain exactly one model ID binding")
job_args = spark_job.args.replace("__REGISTERED_MODEL_ID__", shlex.quote(registered_model.id))
job_name = f"titanic-model-consumption-{uuid.uuid4().hex[:12]}"
output_uri = (
    f"abfss://{FILESYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"{OUTPUT_PREFIX}/{job_name}/"
)

print(f"Job definition: {JOB_PATH}")
print(f"Managed Spark: {spark_job.resources.runtime_version}; size={spark_job.resources.instance_type}")
print(f"Input: {input_uri}")
print(f"Predictions: {output_uri}")

## Model consumption in the managed runtime

The shared [Spark entry point](./src/inference/spark_inference.py) creates the managed session without a local master, loads the staged artifact with `PipelineModel.load`, and calls `model.transform` on distributed input data. Saved imputation, categorical encoding, and Random Forest stages are reused; there is no fitting step. `Survived` is optional for new inference data.

The model copy is sourced from the selected registered version and checksum-verified. This avoids the default Blob account's incompatible DFS endpoint without changing its storage protections.

In [ ]:
final_job = None
if RUN_CONSUMPTION:
    if EXISTING_JOB_NAME:
        submitted_job = ml_client.jobs.get(EXISTING_JOB_NAME)
        if submitted_job.type == "pipeline":
            raise ValueError("Configure a standalone Spark consumption job, not a pipeline")
        print(f"Reusing serverless consumption job: {submitted_job.name}")
    else:
        with DataLakeServiceClient(
            account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net", credential=credential,
        ) as data_lake:
            model_uri = stage_registered_model(
                ml_client, registered_model, credential,
                data_lake.get_file_system_client(FILESYSTEM), STORAGE_ACCOUNT, FILESYSTEM,
            )
        spark_job = load_job(JOB_PATH, params_override=[
            {"name": job_name}, {"args": job_args},
            {"inputs.titanic_data.path": input_uri},
            {"inputs.model_input.path": model_uri},
            {"outputs.predictions.path": output_uri},
            {"tags.registered_model_id": registered_model.id},
        ])
        print(f"Verified registered-model copy: {model_uri}")
        submitted_job = ml_client.jobs.create_or_update(spark_job)
        print(f"Serverless model consumption job: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Model consumption ended with status {final_job.status}")
    print(f"Studio: {final_job.studio_url}")
else:
    print("Submission disabled. Set RUN_TITANIC_SPARK_MODEL_CONSUMPTION=true to consume the model on serverless Spark.")

In [ ]:
if RUN_CONSUMPTION:
    if final_job is None or final_job.status != "Completed":
        raise RuntimeError("A completed Spark job is required to consume its predictions")
    assert_native_spark_job(final_job)
    prediction_output = (final_job.outputs or {}).get("predictions")
    if prediction_output is None or not prediction_output.path:
        raise RuntimeError("The Spark job did not expose its named predictions output")
    download_dir = OUTPUT_ROOT / final_job.name
    download_predictions(prediction_output.path, download_dir, credential)
    source = pd.read_csv(MODULE_ROOT / "data" / "titanic.csv")
    predictions, manifest = load_verified_predictions(
        download_dir, set(source["PassengerId"].astype(int)), registered_model.id, registered_model.path,
    )
    if REFERENCE_PATH:
        reference = pd.read_csv(WORKSHOP_ROOT / REFERENCE_PATH)
        assert_prediction_parity(predictions, reference)
        print("Prediction parity passed against the completed inference pipeline")
    predictions_path = download_dir / "predictions.csv"
    predictions.sort_values("PassengerId").to_csv(predictions_path, index=False)
    print(json.dumps(manifest, indent=2, sort_keys=True))
    display(predictions.sort_values("PassengerId").head(10))
    display(predictions.groupby("prediction").size().rename("passengers").to_frame())
    print(f"Validated serverless model consumption: {len(predictions)} predictions")
    print(f"Predictions CSV: {predictions_path}")
    print(f"Final model consumption status: {final_job.status}")
else:
    print("No Spark execution requested; no predictions were generated or fabricated.")

## Safe reruns

Reset `RUN_TITANIC_SPARK_MODEL_CONSUMPTION=false` after execution. Save the successful standalone job name in `TITANIC_SPARK_MODEL_CONSUMPTION_JOB_NAME` to stream/recheck that run without allocating another Spark cluster. Preserve the actual notebook outputs.

The SDK can read a standalone serverless Spark job back with `compute=None`: its Spark resource configuration and Azure `amlspark` marker identify the managed target. The runtime manifest separately proves Spark 3.5, a non-local master, and registered-model provenance.